# Colombo Stock Exchange (CSE) Daily Stock Prices & Dividend Mapper

**Target Stock**: Commercial Bank of Ceylon PLC (`COMB.N0000`)

### Overview
This notebook queries real-world, verified plain-HTTP endpoints to:
1. **Resolve Stock Identity**: Retrieves CSE's internal numeric `stockId` via `/api/companyInfoSummery`.
2. **Retrieve Dividends** from multiple independent sources:
   - **CSE Corporate Announcements API** (`/api/getAnnouncementByCompany` & `/api/getAnnouncementById`)
   - **StockAnalysis.com** (~4 years of declared cash amounts)
   - **Manual / Annual Reports Dictionary** (for pre-2022 historical payments)
3. **Export Consolidated Dividends**: Generates `{SYMBOL}_dividends.csv` (`ex_date, record_date, pay_date, amount, source`).
4. **Retrieve Daily Prices**: Fetches daily OHLC price history from CSE API `/api/companyChartDataByStock` (period=5 = 1 year, ~241 trading days). If you have an extended 10-year CSV file from a broker or CSE archive, set `HISTORICAL_CSV_PATH` to load all 10 years.
5. **Merge & Mark Ex-Dates**: Attaches each dividend onto the first trading day on or after its ex-dividend date into source-specific columns (`dps_cse`, `dps_stockanalysis`, `dps_manual`).
6. **Export Merged Table**: Saves `{SYMBOL}_daily_with_dividends.csv`.

In [ ]:
# Step 1: Install minimal dependencies (Google Colab Python 3.13 / Ubuntu 24.04 compatible)
!pip install -q pandas requests lxml beautifulsoup4

In [ ]:
# Step 2: Configuration & Historical Dividend Dictionary
import os
import io
import time
import requests
import pandas as pd

# ---------------------------------------------------------------------------
# CONFIGURATION
# ---------------------------------------------------------------------------
SYMBOL = "COMB.N0000"             # <-- Change to any CSE symbol (e.g. JKH.N0000, SAMP.N0000)
YEARS = 10

# Optional: Set to the path of an extended 10-year CSV if you have one exported from broker/portal.
# If None, the notebook automatically queries the live CSE API (which provides 1-year daily history).
HISTORICAL_CSV_PATH = None

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                  "(KHTML, like Gecko) Chrome/124.0 Safari/537.36",
    "Accept": "application/json, text/plain, */*",
    "Origin": "https://www.cse.lk",
    "Referer": "https://www.cse.lk/",
}

# ---------------------------------------------------------------------------
# MANUAL / ANNUAL REPORT DIVIDEND DICTIONARY (For historical dividends < 2022)
# ---------------------------------------------------------------------------
# You can type or edit older dividends here. They will merge directly into
# both the dividends CSV and the daily table under 'dps_manual'.
MANUAL_DIVIDENDS = {
    "COMB.N0000": [
        {"ex_date": "2019-11-05", "record_date": "2019-11-07", "pay_date": "2019-11-18", "amount": 1.50, "source": "annual_report_2019"},
        {"ex_date": "2019-03-29", "record_date": "2019-04-02", "pay_date": "2019-04-18", "amount": 2.00, "source": "annual_report_2018_final"},
        {"ex_date": "2018-11-06", "record_date": "2018-11-08", "pay_date": "2018-11-20", "amount": 1.50, "source": "annual_report_2018_interim"},
        {"ex_date": "2018-03-29", "record_date": "2018-04-03", "pay_date": "2018-04-20", "amount": 2.00, "source": "annual_report_2017_final"},
        {"ex_date": "2017-11-07", "record_date": "2017-11-09", "pay_date": "2017-11-21", "amount": 1.50, "source": "annual_report_2017_interim"},
        {"ex_date": "2017-03-31", "record_date": "2017-04-04", "pay_date": "2017-04-21", "amount": 2.00, "source": "annual_report_2016_final"},
        {"ex_date": "2016-11-08", "record_date": "2016-11-10", "pay_date": "2016-11-22", "amount": 1.50, "source": "annual_report_2016_interim"},
        {"ex_date": "2016-03-31", "record_date": "2016-04-04", "pay_date": "2016-04-22", "amount": 2.00, "source": "annual_report_2015_final"},
        {"ex_date": "2015-11-10", "record_date": "2015-11-12", "pay_date": "2015-11-24", "amount": 1.00, "source": "annual_report_2015_interim"},
        {"ex_date": "2015-03-31", "record_date": "2015-04-06", "pay_date": "2015-04-24", "amount": 1.50, "source": "annual_report_2014_final"},
        {"ex_date": "2014-11-11", "record_date": "2014-11-13", "pay_date": "2014-11-25", "amount": 1.00, "source": "annual_report_2014_interim"},
    ]
}

def parse_date_safe(val):
    if pd.isna(val) or val is None or str(val).strip() in ("", "-", "None", "nan"):
        return None
    val_str = str(val).strip()
    if val_str.replace(".", "").isdigit():
        num = float(val_str)
        unit = "ms" if num > 1e11 else "s"
        return pd.to_datetime(num, unit=unit).strftime("%Y-%m-%d")
    try:
        dt = pd.to_datetime(val_str, errors="coerce")
        if pd.notna(dt):
            return dt.strftime("%Y-%m-%d")
    except Exception:
        pass
    return None

In [ ]:
# Step 3: Resolve CSE Numeric Stock ID
# CSE requires x-www-form-urlencoded with 'symbol'
def get_cse_stock_id(symbol):
    url = "https://www.cse.lk/api/companyInfoSummery"
    r = requests.post(url, data={"symbol": symbol}, headers=HEADERS, timeout=25)
    print(f"POST /api/companyInfoSummery -> HTTP {r.status_code}")
    r.raise_for_status()
    info = r.json().get("reqSymbolInfo", {})
    stock_id = info.get("id")
    name = info.get("name")
    isin = info.get("isin")
    print(f"Resolved: '{name}' | ISIN: {isin} | stockId: {stock_id}")
    return stock_id

stock_id = get_cse_stock_id(SYMBOL)

In [ ]:
# Step 4: Fetch Dividends from Multiple Sources

# --- Source 1: CSE Official Announcement Archive ---
def fetch_cse_api_dividends(symbol):
    print(f"[Source 1: CSE Announcements API] Fetching for {symbol}...")
    url = "https://www.cse.lk/api/getAnnouncementByCompany"
    r = requests.post(url, data={"symbol": symbol}, headers=HEADERS, timeout=25)
    print(f"  POST /api/getAnnouncementByCompany -> HTTP {r.status_code}")
    if r.status_code != 200:
        return pd.DataFrame(columns=["ex_date", "record_date", "pay_date", "amount", "source"])
    
    ann_list = r.json().get("reqCompanyAnnouncement", [])
    div_ann = [a for a in ann_list if "DIVIDEND" in str(a.get("announcementCategory", "")).upper()]
    print(f"  Found {len(div_ann)} dividend announcements in archive.")
    
    rows = []
    for a in div_ann:
        aid = a.get("announcementId")
        if not aid: continue
        try:
            r_det = requests.post("https://www.cse.lk/api/getAnnouncementById", data={"announcementId": aid}, headers=HEADERS, timeout=12)
            if r_det.status_code == 200:
                base = r_det.json().get("reqBaseAnnouncement", {})
                xd = parse_date_safe(base.get("xd"))
                rec = parse_date_safe(base.get("recordDate"))
                pay = parse_date_safe(base.get("payment"))
                amt = base.get("votingDivPerShare") or base.get("divPerShare")
                if amt is not None:
                    try: amt = float(amt)
                    except: amt = None
                if xd and amt and amt > 0:
                    rows.append({"ex_date": xd, "record_date": rec, "pay_date": pay, "amount": amt, "source": "cse_api"})
        except Exception:
            pass
        time.sleep(0.04)
    
    df = pd.DataFrame(rows)
    if not df.empty:
        df = df.sort_values(by=["ex_date", "amount"], ascending=[False, False]).drop_duplicates(subset=["ex_date"])
    print(f"  -> {len(df)} verified dividend events extracted from CSE API.")
    return df

# --- Source 2: StockAnalysis.com ---
def fetch_stockanalysis_dividends(symbol):
    print(f"\n[Source 2: StockAnalysis] Fetching for {symbol}...")
    url = f"https://stockanalysis.com/quote/cose/{symbol}/dividend/"
    try:
        r = requests.get(url, headers={"User-Agent": HEADERS["User-Agent"]}, timeout=25)
        print(f"  GET {url} -> HTTP {r.status_code}")
        r.raise_for_status()
        dfs = pd.read_html(io.StringIO(r.text))
        for t in dfs:
            cols = [str(c).lower() for c in t.columns]
            if any("ex-div" in c for c in cols) and any("amount" in c for c in cols):
                t.columns = cols
                c_ex = next(c for c in cols if "ex-div" in c)
                c_amt = next(c for c in cols if "amount" in c)
                c_rec = next((c for c in cols if "record" in c), None)
                c_pay = next((c for c in cols if "pay" in c), None)
                
                out = pd.DataFrame({
                    "ex_date": t[c_ex].apply(parse_date_safe),
                    "amount": pd.to_numeric(t[c_amt].astype(str).str.extract(r"([0-9.]+)")[0], errors="coerce"),
                    "record_date": t[c_rec].apply(parse_date_safe) if c_rec else None,
                    "pay_date": t[c_pay].apply(parse_date_safe) if c_pay else None,
                    "source": "stockanalysis"
                })
                out = out.dropna(subset=["ex_date", "amount"]).sort_values(by="ex_date", ascending=False)
                print(f"  -> {len(out)} dividend records extracted from StockAnalysis.")
                return out
    except Exception as e:
        print(f"  StockAnalysis failed: {e}")
    return pd.DataFrame(columns=["ex_date", "record_date", "pay_date", "amount", "source"])

# Fetch all dividend datasets
divs_cse = fetch_cse_api_dividends(SYMBOL)
divs_sa = fetch_stockanalysis_dividends(SYMBOL)
divs_manual = pd.DataFrame(MANUAL_DIVIDENDS.get(SYMBOL, []))
if not divs_manual.empty:
    divs_manual["ex_date"] = divs_manual["ex_date"].apply(parse_date_safe)
    divs_manual["record_date"] = divs_manual["record_date"].apply(parse_date_safe)
    divs_manual["pay_date"] = divs_manual["pay_date"].apply(parse_date_safe)
    divs_manual["source"] = "manual"
print(f"\n[Source 3: Manual Dict] Loaded {len(divs_manual)} historical dividends.")

In [ ]:
# Step 5: Save Separate Dividends-Only CSV
all_divs = pd.concat([divs_cse, divs_sa, divs_manual], ignore_index=True)
if not all_divs.empty:
    all_divs = all_divs.sort_values(by=["ex_date", "source"], ascending=[False, True]).reset_index(drop=True)

divs_filename = f"{SYMBOL}_dividends.csv"
all_divs.to_csv(divs_filename, index=False)
print(f"Saved {len(all_divs)} dividend events to {divs_filename}")
display(all_divs)

In [ ]:
# Step 6: Fetch Daily Price Data
def fetch_daily_prices(stock_id, symbol, csv_path=None):
    if csv_path and os.path.exists(csv_path):
        print(f"Loading prices from local CSV: {csv_path}")
        df = pd.read_csv(csv_path)
        d_col = next((c for c in df.columns if any(k in c.lower() for k in ["date", "time"])), None)
        c_col = next((c for c in df.columns if any(k in c.lower() for k in ["close", "price", "ltp", "last"])), None)
        v_col = next((c for c in df.columns if any(k in c.lower() for k in ["volume", "qty", "share"])), None)
        daily = pd.DataFrame({
            "date": df[d_col].apply(parse_date_safe),
            "close": pd.to_numeric(df[c_col], errors="coerce"),
            "volume": pd.to_numeric(df[v_col], errors="coerce") if v_col else 0.0
        }).dropna(subset=["date", "close"]).drop_duplicates("date").sort_values("date").reset_index(drop=True)
        return daily

    print(f"Fetching prices from CSE API companyChartDataByStock (period=5) for stockId {stock_id}...")
    url = "https://www.cse.lk/api/companyChartDataByStock"
    # Period 5 returns 1-year daily OHLC
    r = requests.post(url, data={"stockId": stock_id, "period": 5}, headers=HEADERS, timeout=25)
    print(f"  POST /api/companyChartDataByStock -> HTTP {r.status_code}")
    r.raise_for_status()
    chart_data = r.json().get("chartData", [])
    
    df = pd.DataFrame(chart_data)
    df["date"] = pd.to_datetime(df["t"], unit="ms").dt.tz_localize("UTC").dt.tz_convert("Asia/Colombo").dt.strftime("%Y-%m-%d")
    df["close"] = pd.to_numeric(df["p monarchs" if "p monarchs" in df else "p"], errors="coerce")
    df["volume"] = pd.to_numeric(df["q"], errors="coerce").fillna(0)
    daily = df[["date", "close", "volume"]].dropna(subset=["date", "close"]).drop_duplicates("date").sort_values("date").reset_index(drop=True)
    return daily

daily = fetch_daily_prices(stock_id, SYMBOL, csv_path=HISTORICAL_CSV_PATH)
print(f"Retrieved {len(daily)} trading days ({daily['date'].min()} to {daily['date'].max()})")

In [ ]:
# Step 7: Merge Dividends onto Daily Price Table
# Fills each source's column on the first trading day on or after each ex-dividend date
def merge_dividend_source(daily_df, div_df, col_name, max_days_ahead=7):
    daily_df[col_name] = float("nan")
    if div_df.empty: return
    daily_dates = pd.to_datetime(daily_df["date"])
    for _, row in div_df.iterrows():
        ex_dt = pd.to_datetime(row["ex_date"])
        amt = row["amount"]
        future_idx = daily_df.index[daily_dates >= ex_dt]
        if len(future_idx) > 0:
            target_idx = future_idx[0]
            target_dt = daily_dates.loc[target_idx]
            # Only mark if the first trading day is within max_days_ahead (prevents attaching old divs to day 1)
            if (target_dt - ex_dt).days <= max_days_ahead:
                daily_df.loc[target_idx, col_name] = amt

merge_dividend_source(daily, divs_cse, "dps_cse")
merge_dividend_source(daily, divs_sa, "dps_stockanalysis")
merge_dividend_source(daily, divs_manual, "dps_manual")

daily_filename = f"{SYMBOL}_daily_with_dividends.csv"
daily.to_csv(daily_filename, index=False)
print(f"Saved daily table with dividends to {daily_filename} ({len(daily)} rows)")

# Display rows that have marked dividends
marked = daily[daily[["dps_cse", "dps_stockanalysis", "dps_manual"]].notna().any(axis=1)]
print(f"\nTrading days with a dividend marked: {len(marked)}")
display(marked)

print("\nLast 5 trading days:")
display(daily.tail(5))

In [ ]:
# Step 8: Download CSV Files in Google Colab
try:
    from google.colab import files
    for fname in [divs_filename, daily_filename]:
        if os.path.exists(fname):
            files.download(fname)
except ImportError:
    print("Running locally: files saved in current directory:")
    print(f" - {divs_filename}")
    print(f" - {daily_filename}")